Step 0 — Set up in five minutes, not an hour. Use the Chinook database: it's a fake digital music store (customers, invoices, tracks, artists) with enough tables to require real joins. It ships as a single SQLite file, so the whole setup is downloading chinook.db and running sqlite3 chinook.db — no server, no Docker. If the group prefers a browser, sqliteonline.com or DB Browser for SQLite work fine. Postgres is worth it only if you're doing the stretch, since its EXPLAIN ANALYZE output is much richer.

Step 1 — Sketch the schema before writing any query. Spend ten minutes drawing the tables and their foreign keys on paper: which table connects customers to purchases? Where does a track's genre live? Every wrong query in this challenge comes from a wrong mental model of the schema, not from syntax. .schema in sqlite3 dumps it all.

Step 2 — The 10 questions, in escalating order. Here's a ready-made ladder; each rung introduces exactly one new concept:

How many customers are there per country? (GROUP BY, COUNT)
What are the 10 longest tracks? (ORDER BY, LIMIT)
Total revenue per year? (date functions, SUM)
Which tracks have never been purchased? (LEFT JOIN … IS NULL, or NOT IN)
Revenue per genre, highest first? (two joins plus aggregation)
Which countries have more than 40 invoices? (HAVING vs WHERE — the classic trap)
Average invoice total per customer, but only customers with 5+ invoices? (aggregate of an aggregate — needs a subquery or CTE)
Top-spending customer in each country? (correlated subquery, or a window function if someone's ahead of the curve)
Which artists have tracks in the most playlists? (a chain of 3–4 joins)
Month-over-month revenue growth as a percentage? (self-join on shifted months, or LAG — a natural bridge into the stretch)

For a mixed group, everyone attempts 1–6 solo; 7–10 work well in pairs. The rule that makes this a sprint: before running each query, say out loud what the result should roughly look like ("about 25 rows, one per country, biggest number first"). Predicting output is the skill; running queries is free.

Stretch part 1 — Window functions. The mental shift: GROUP BY collapses rows, window functions keep every row and add context alongside it. Redo question 8 with RANK() OVER (PARTITION BY country ORDER BY total DESC) and question 10 with LAG(revenue) OVER (ORDER BY month) — comparing your earlier subquery solutions to these makes the concept click. Then add a running total of revenue over time (SUM(...) OVER (ORDER BY invoice_date)), which is the pattern behind every cumulative chart you'll ever build.


Stretch part 2 — Make a slow query fast with EXPLAIN. First you need a slow query, and Chinook is too small to be slow — so generate pain: create a table with a million synthetic invoice rows (a recursive CTE or a quick Python loop does it), then query it with a filter on a non-indexed column. Run EXPLAIN QUERY PLAN (SQLite) or EXPLAIN ANALYZE (Postgres) and find the phrase that matters: SCAN — meaning the engine reads every row. Add an index on the filtered column, run it again, and watch it flip to SEARCH ... USING INDEX with a wall-clock drop you can feel. Then the deeper lesson: rewrite a query so the index becomes useless again — for instance, wrapping the indexed column in a function like WHERE lower(email) = … — and watch it degrade. Knowing what breaks index usage is worth more than knowing indexes exist.